<a href="https://colab.research.google.com/github/klizaola/Predicting-Pollinator-Syndrome/blob/main/PollinatorProject_week1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn, scipy
!pip install prince
from sklearn import model_selection
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
from scipy.cluster.hierarchy import linkage, dendrogram
from IPython.display import display
from pathlib import Path
import json

print("NumPy", np.__version__, "scikit-learn", sklearn.__version__, "SciPy", scipy.__version__)
plt.rcParams.update({'figure.dpi': 110, 'font.size': 11, 'axes.spines.top': False, 'axes.spines.right': False})
COLORS = ['#0072B2', '#D55E00', '#009E73', '#CC79A7', '#E69F00', '#56B4E9']
OUT = Path('unsupervised_outputs'); OUT.mkdir(exist_ok=True)

def finish(name):
    plt.tight_layout()
    plt.savefig(OUT / (name + '.png'), dpi=160, bbox_inches='tight')
    plt.show()

def scatter_groups(ax, points, labels=None, title='', axis_labels=('PC1', 'PC2')):
    if labels is None:
        ax.scatter(points[:, 0], points[:, 1], color=COLORS[0], s=30, alpha=.8)
    else:
        for i, label in enumerate(np.unique(labels)):
            take = np.asarray(labels) == label
            ax.scatter(points[take, 0], points[take, 1], color=COLORS[i % len(COLORS)], s=30, alpha=.8, label=str(label))
        ax.legend(fontsize=9)
    ax.set(xlabel=axis_labels[0], ylabel=axis_labels[1], title=title)

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass
try:
    import ipywidgets as widgets
    HAVE_WIDGETS = True
except ImportError:
    HAVE_WIDGETS = False
    print('Sliders unavailable: use the function calls provided below.')

NumPy 2.4.6 scikit-learn 1.6.1 SciPy 1.16.3


In [2]:
from google.colab import files
uploaded = files.upload("Dryad dataset_final_clean.csv")
dryad_data = pd.read_csv(next(iter(uploaded)))


Saving Dryad dataset_final_clean.csv to Dryad datset_clean.csv/Dryad dataset_final_clean.csv


In [3]:
dryad_cols = ['Family', 'Floral visitor', 'Plant species', 'Isolate1', 'Isolate2', 'Isolate3', 'Isolate4', 'Isolate5']

for col in dryad_cols:
    print(f"=== Class Counts for '{col}' ===")
    counts = dryad_data[col].value_counts()
    percentages = dryad_data[col].value_counts(normalize=True) * 100

    distribution = pd.DataFrame({
        'Count': counts,
        'Percentage (%)': percentages.round(2)
    })
    display(distribution.head(10))
    print("\n")

=== Class Counts for 'Family' ===


,Count,Percentage (%)
Family,,
Proteaceae,102,56.04
Iridaceae,19,10.44
Ericaceae,14,7.69
Orobanchaceae,9,4.95
Acanthaceae,7,3.85
Xanthorreaceae,6,3.30
Lamiaceae,6,3.30
Scrophulariaceae,6,3.30
Caryophyllaceae,3,1.65




=== Class Counts for 'Floral visitor' ===


,Count,Percentage (%)
Floral visitor,,
Beetle,102,56.04
Long-tongued insects,38,20.88
Birds,25,13.74
Short-tongued insects,17,9.34




=== Class Counts for 'Plant species' ===


,Count,Percentage (%)
Plant species,,
Protea welwitschii,21,11.54
Protea caffra,14,7.69
Protea roupelliae,13,7.14
Protea dracomontana,13,7.14
Protea subvestita,12,6.59
Protea repens,10,5.49
Protea laurifolia,10,5.49
Glumicalyx goseloides,6,3.30
Protea neriifolia,6,3.30


... (6 outputs left out)



=== Class Counts for 'Isolate4' ===


,Count,Percentage (%)
Isolate4,,
0,166,91.21
Klebsiella_Enterobacter,3,1.65
Hanseniaspora thailandica,2,1.10
Metschnikowia proteae,1,0.55
Rhodotorula taiwanensis,1,0.55
Rosenbergiella,1,0.55
Wickerhamomyces,1,0.55
Paraburkholderia,1,0.55
Rhodotorula mucilaginosa,1,0.55




=== Class Counts for 'Isolate5' ===


,Count,Percentage (%)
Isolate5,,
0,178,97.80
Pseudomonas_2,1,0.55
Pichia,1,0.55
Asaia,1,0.55
Rhodotorula mucilaginosa,1,0.55


In [4]:
import pandas as pd

from sklearn.preprocessing import MultiLabelBinarizer


df = dryad_data.copy()


df.columns = df.columns.str.strip()


isolate_columns = [
    "Isolate1",
    "Isolate2",
    "Isolate3",
    "Isolate4"
]


df["Isolates"] = df[isolate_columns].apply(
    lambda row: [
        str(isolate).strip() for isolate in row
        if pd.notna(isolate) and isolate != 0 and str(isolate).strip() != "0" and str(isolate).strip() != ""
    ],
    axis=1
)


mlb = MultiLabelBinarizer()

encoded_isolates = pd.DataFrame(
    mlb.fit_transform(df["Isolates"]),
    columns=mlb.classes_,
    index=df.index
)

mlb = MultiLabelBinarizer()

encoded_isolates = pd.DataFrame(
    mlb.fit_transform(df["Isolates"]),
    columns=mlb.classes_,
    index=df.index
)


X = encoded_isolates

y = df["Floral visitor"]

print(X)
print(y)



     Achromobacter  Acinetobacter  Arthrobacter  Asaia  Auriculibuller fuscus  \
0                0              0             0      0                      0   
1                0              0             0      0                      0   
2                0              0             0      0                      0   
3                0              1             0      0                      0   
4                0              0             0      1                      0   
..             ...            ...           ...    ...                    ...   
177              0              0             0      0                      0   
178              0              0             0      0                      0   
179              0              0             0      0                      0   
180              0              0             0      0                      0   
181              0              0             0      0                      0   

     Bacillus_1  Bacillus_2

In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.40, random_state=42, stratify=y
)

X_validation, X_test, y_validation, y_test = train_test_split(
    X_test, y_test, test_size=0.50, random_state=42, stratify=y_40
)
print(f"Training set size:   {X_train.shape[0]} samples ({X_train.shape[0] / len(X) * 100:.1f}%)")
print(f"Validation set size: {X_validation.shape[0]} samples ({X_validation.shape[0] / len(X) * 100:.1f}%)")
print(f"Test set size:       {X_test.shape[0]} samples ({X_test.shape[0] / len(X) * 100:.1f}%)")

Training set size:   109 samples (59.9%)
Validation set size: 36 samples (19.8%)
Test set size:       37 samples (20.3%)


In [13]:
print(X_test)

     Achromobacter  Acinetobacter  Arthrobacter  Asaia  Auriculibuller fuscus  \
162              0              0             0      0                      0   
36               0              0             0      0                      0   
126              0              0             0      0                      0   
179              0              0             0      0                      0   
163              0              0             0      0                      0   
79               0              0             0      0                      0   
175              0              0             0      0                      0   
146              0              0             0      0                      0   
150              0              0             0      0                      0   
124              0              0             0      0                      0   
169              0              0             0      0                      0   
28               0          

In [10]:
print(df['Plant species'].nunique())

43


In [17]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split, GroupKFold
from sklearn.ensemble import RandomForestClassifier

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.40, random_state=42, stratify=y
)

X_validation, X_test, y_validation, y_test = train_test_split(
    X_test, y_test, test_size=0.50, random_state=42, stratify=y_test
)

groups = df.loc[X_validation.index, "Plant species"].values
n_splits = min(len(np.unique(groups)), 5)
group_kfold = GroupKFold(n_splits=n_splits)

most_frequent_class = y_train_full.mode()[0]
simple_baseline = np.full_like(y_test, fill_value=most_frequent_class, dtype=object)

cv_scores = []
for train_idx, test_idx in group_kfold.split(X_validation, y_validation, groups=groups):
    X_tr, X_val = X_validation.iloc[train_idx], X_validation.iloc[test_idx]
    y_tr, y_val = y_validation.iloc[train_idx], y_validation.iloc[test_idx]

    rf_fold = RandomForestClassifier(
        n_estimators=100,
        random_state=36,
        max_depth=None,
        max_features='sqrt',
        max_leaf_nodes=10,
        class_weight='balanced'
    )
    rf_fold.fit(X_tr, y_tr)
    preds = rf_fold.predict(X_val)

    acc = accuracy_score(y_val, preds)
    f1 = f1_score(y_val, preds, average='macro')
    cv_scores.append({'fold': len(cv_scores), 'acc': acc, 'f1': f1})

random_forest = RandomForestClassifier(
    n_estimators=100,
    random_state=36,
    max_depth=None,
    max_features='sqrt',
    max_leaf_nodes=10,
    class_weight='balanced'
)
random_forest.fit(X_train_full, y_train_full)

proba = random_forest.predict_proba(X_test)
predictions = random_forest.predict(X_test)

accuracy = accuracy_score(y_test, predictions)
f1 = f1_score(y_test, predictions, average='macro')

baseline_accuracy = accuracy_score(y_test, simple_baseline)
baseline_f1 = f1_score(y_test, simple_baseline, average='macro')

print(f"Baseline Accuracy: {baseline_accuracy:.4f} | Baseline F1: {baseline_f1:.4f}")
for score in cv_scores:
    print(f"Fold {score['fold']}: Accuracy = {score['acc']:.4f}, F1 = {score['f1']:.4f}")
print(f"Test Set Accuracy: {accuracy:.4f} | Test Set F1: {f1:.4f}")

Baseline Accuracy: 0.5676 | Baseline F1: 0.1810
Fold 0: Accuracy = 0.3750, F1 = 0.2815
Fold 1: Accuracy = 0.8571, F1 = 0.7000
Fold 2: Accuracy = 0.7143, F1 = 0.3889
Fold 3: Accuracy = 0.7143, F1 = 0.6250
Fold 4: Accuracy = 0.8571, F1 = 0.5556
Test Set Accuracy: 0.7568 | Test Set F1: 0.6018
